# 예제 03. 서버 바꿔 쓰기 — 학과 서버와 Gemini · Example 03 — Switching Servers: the Department Server and Gemini

전산실험 2 · AI Agent 만들기 · **1주차** · 예상 소요 25분
Computational Experiments 2 · Building AI Agents · **Week 1** · about 25 minutes

## 목표 · Objectives
- 하나의 코드로 두 개의 LLM 서버를 번갈아 쓴다
- OpenAI 호환 규격이 무엇을 표준화했는지 이해한다
- 소형 모델과 대형 모델의 응답 차이를 직접 비교한다

> **EN**
> - Drive two different LLM servers from one piece of code
> - Understand what the OpenAI-compatible protocol actually standardises
> - Compare a small model against a large one side by side

## 왜 필요한가 · Why this matters
1. **실용**: 학과 서버가 점검 중이어도 집에서 복습할 수 있어야 합니다
2. **감각**: 앞으로 우리가 상대할 소형 모델의 수준을 알아야 설계를 제대로 합니다

> **EN**
> 1. **Practical** — you must be able to revise at home even while the
>    department server is down for maintenance.
> 2. **Calibration** — you can only design well once you know what the small
>    model we will be working with is actually capable of.

---
## 0. Gemini 무료 키 발급 (5분) · Getting a free Gemini key (5 minutes)

1. https://aistudio.google.com/apikey 접속 (Google 계정 로그인)
2. **Create API key** 클릭
3. 발급된 키를 복사
4. Colab Secrets 에 이름 `GEMINI_API_KEY` 로 저장 + 이 노트북 액세스 토글 켜기

> 무료 등급에는 분당 호출 수 제한이 있습니다.
> 2주차 이후의 반복 실험은 되도록 학과 서버로 하세요.

> **EN**
> 1. Go to https://aistudio.google.com/apikey and sign in with a Google account
> 2. Click **Create API key**
> 3. Copy the key you are given
> 4. Save it in Colab Secrets as `GEMINI_API_KEY` and turn on notebook access
>
> The free tier limits calls per minute, so from week 2 on, prefer the
> department server for the repeated experiments.

In [ ]:
!pip install -q openai

import os
from openai import OpenAI

def get_secret(name, secret=True):
    # Colab Secrets -> 환경변수 -> 직접 입력. 값은 코드에 적지 않는다.
    # EN — Colab Secrets -> environment variable -> prompt. Never write the value here.
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v.strip()
    except Exception:
        pass
    if os.environ.get(name):
        return os.environ[name].strip()
    if secret:
        import getpass
        v = getpass.getpass(f"{name}: ").strip()
    else:
        v = input(f"{name}: ").strip()
    os.environ[name] = v
    return v


def get_base_url():
    url = get_secret("KMU_BASE_URL", secret=False).rstrip("/")
    for suffix in ("/chat/completions", "/completions"):
        if url.endswith(suffix):
            url = url[: -len(suffix)]
    return url


get_key = get_secret          # 1주차에 쓰던 이름 유지 / keep the week-1 name
print("준비 완료")

---
## 1. 두 서버의 설정을 표로 정리한다 · Putting both servers in one table

핵심은 **바꾸는 것이 세 가지뿐** 이라는 점입니다: 주소, 모델 이름, 키.
라이브러리도 호출 방식도 똑같습니다. 이것이 "OpenAI 호환"의 의미입니다.

> **EN** — The point is that **only three things change**: the URL, the model
> name and the key. The library and the call are identical. That is what
> "OpenAI-compatible" means.

In [ ]:
PROVIDERS = {
    "kmu": {
        "base_url": get_base_url(),        # Secrets 에서 읽습니다 / read from Secrets
        "model":    "gemma4",
        "key_name": "KMU_API_KEY",
        "label":    "계명대 학과 서버 (소형 모델)",
    },
    "gemini": {
        "base_url": "https://generativelanguage.googleapis.com/v1beta/openai/",
        "model":    "gemini-2.0-flash",
        "key_name": "GEMINI_API_KEY",
        "label":    "Google Gemini (대형 모델, 백업)",
    },
}

def make_client(provider):
    conf = PROVIDERS[provider]
    client = OpenAI(api_key=get_key(conf["key_name"]),
                    base_url=conf["base_url"], timeout=120)
    return client, conf["model"], conf["label"]


def ask_with(provider, prompt, **kw):
    client, model, _ = make_client(provider)
    r = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        **kw,
    )
    return r.choices[0].message.content.strip()

print("설정 완료")

---
## 2. 스위치 하나로 바꿔 쓰기 · One switch to change servers

앞으로 모든 노트북 위쪽에 이 스위치를 둡니다.
학과 서버가 안 되면 `PROVIDER` 를 `"gemini"` 로 바꾸기만 하면 됩니다.

> **EN** — From here on, every notebook carries this switch near the top. If the
> department server is unavailable, just set `PROVIDER` to `"gemini"`.

In [ ]:
PROVIDER = "kmu"        # <-- 안 되면 "gemini" 로 바꾸세요 / switch to "gemini" if this fails

client, MODEL, LABEL = make_client(PROVIDER)
print("사용 중:", LABEL, "|", MODEL)

def ask(prompt, system=None, **kw):
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    r = client.chat.completions.create(model=MODEL, messages=messages, **kw)
    return r.choices[0].message.content.strip()

print()
print(ask("자유도가 무엇인지 두 문장으로 설명해 주세요."))

---
## 3. 소형 모델과 대형 모델을 나란히 비교 · Small model vs large model, side by side

같은 프롬프트를 두 서버에 던져 봅시다.
**특히 형식 지시를 얼마나 잘 지키는지** 를 눈여겨보세요.
이것이 앞으로 우리를 계속 괴롭힐 문제입니다.

> **EN** — Send the same prompt to both servers. Watch **how well each one obeys
> the format instruction** in particular: that is the problem that will follow
> us for the rest of the course.

In [ ]:
def 비교(prompt, **kw):
    # EN — Run the same prompt against both providers and print them together.
    for p in ["kmu", "gemini"]:
        print("=" * 64)
        print(f"[{PROVIDERS[p]['label']}]")
        print("-" * 64)
        try:
            print(ask_with(p, prompt, **kw))
        except Exception as exc:
            print(f"(호출 실패: {type(exc).__name__}: {exc})")
        print()


비교("귀무가설과 대립가설의 차이를 두 문장으로 설명해 주세요.", max_tokens=300)

### 형식 지시를 지키는가 — 여기가 진짜 시험대 · Does it obey the format? The real test

에이전트는 모델의 출력을 **코드로 파싱**해야 합니다.
모델이 "네, 알겠습니다! 다음은 요청하신 JSON입니다:" 같은 말을 앞에 붙이면
파싱이 곧바로 실패합니다.

> **EN** — An agent has to **parse the model's output in code**. The moment the
> model prefixes its answer with "Sure! Here is the JSON you asked for:", the
> parse fails outright.

In [ ]:
형식지시 = (
    "다음 변수들을 분류해서 JSON으로만 답하세요. 다른 말은 절대 하지 마세요.\n"
    '형식: {"연속형": [...], "범주형": [...]}\n'
    "변수: 나이, 성별, 소득, 혈액형, 키, 학력수준"
)

비교(형식지시, temperature=0.0, max_tokens=300)

In [ ]:
# 실제로 파싱해 보기 -- 성공하는가?
# EN — Now actually try to parse it. Does it succeed?
import json

for p in ["kmu", "gemini"]:
    print(f"[{p}]", end=" ")
    try:
        raw = ask_with(p, 형식지시, temperature=0.0, max_tokens=300)
        parsed = json.loads(raw)
        print("파싱 성공 ->", parsed)
    except json.JSONDecodeError as exc:
        print("파싱 실패!")
        print("     원문:", repr(raw[:120]))
        print("     오류:", exc)
    except Exception as exc:
        print("호출 실패:", type(exc).__name__)

> **소형 모델은 높은 확률로 파싱에 실패합니다.**
> 코드펜스(``` ```json ```)로 감싸거나, 설명을 덧붙이거나, 따옴표를 빠뜨립니다.
>
> 이것은 모델을 탓할 일이 아니라 **우리가 해결해야 할 공학 문제**입니다.
> 3주차 전체가 이 문제를 다룹니다. 오늘은 "이런 일이 일어난다"만 기억하세요.

> **EN — the small model will very often fail to parse.** It wraps the answer in
> a code fence, adds an explanation, or drops a quote.
>
> This is not the model's fault to complain about; it is **an engineering problem
> for us to solve**. The whole of week 3 is devoted to it. For today, just
> remember that this happens.

---
## 연습문제 · Exercises

**문제 1.** 같은 프롬프트를 두 서버에 각각 10회 던져,
JSON 파싱 **성공률**을 계산하고 비교하시오.

**문제 2.** 프롬프트를 고쳐서 소형 모델의 파싱 성공률을 올려 보시오.
어떤 문구가 효과가 있었는가?
(힌트: 예시를 함께 주기, "```" 를 쓰지 말라고 명시하기, 시작 문자를 지정하기)

**문제 3.** 두 서버의 응답 속도를 각 5회씩 측정해 비교하시오. (`time.perf_counter`)

> **EN — 1.** Send the same prompt to each server 10 times and compute and
> compare the JSON parse **success rate**.
>
> **EN — 2.** Revise the prompt to raise the small model's parse rate. Which
> phrasings worked? (Hint: give an example, state explicitly that "```" must not
> be used, pin down the first character.)
>
> **EN — 3.** Measure and compare the response latency of both servers over five
> runs each (`time.perf_counter`).

In [ ]:
# 여기에 직접 작성해 보세요 / Write your own code here

<details>
<summary><b>해설 보기 · Show the solutions</b></summary>

**문제 1.** 예시 코드
```python
def 성공률(p, prompt, n=10):
    성공 = 0
    for _ in range(n):
        try:
            json.loads(ask_with(p, prompt, temperature=0.0, max_tokens=300))
            성공 += 1
        except Exception:
            pass
    return 성공 / n

for p in ["kmu", "gemini"]:
    print(p, 성공률(p, 형식지시))
```
n=10 은 표준오차가 커서 두 모델의 차이를 확신하기 어렵습니다.
**표본을 늘리거나 비율 검정을 해야 합니다.** 11주차의 주제입니다.

**문제 2.** 효과가 큰 순서대로
1. **출력 예시를 하나 보여주기** (few-shot) — 가장 효과가 큽니다
2. `설명, 인사, 코드블록 표시(백틱)를 쓰지 마세요` 처럼 **금지 사항을 명시**
3. `응답은 반드시 { 로 시작해서 } 로 끝나야 합니다` 처럼 **시작·끝 지정**
4. `temperature=0.0`

그래도 완벽해지지는 않습니다. 그래서 3주차에는 프롬프트를 고치는 데서 멈추지 않고,
**받아낸 뒤 파싱·검증·재시도하는 코드**를 만듭니다.

**문제 3.** 학과 서버는 모델이 작아 대체로 빠르지만, 동시 접속자가 많으면 느려집니다.
수업 중 30명이 동시에 호출한다는 점을 감안하세요.

---

**EN — 1.** With n = 10 the standard error is large, so you cannot be confident
about the difference between the two models. **Increase the sample or run a
proportion test** — the subject of week 11.

**EN — 2.** In descending order of effect: (1) show one output example
(few-shot), which helps most; (2) state the prohibitions explicitly, as in "do
not write explanations, greetings or code-fence backticks"; (3) pin down the
first and last character — "the response must begin with { and end with }";
(4) `temperature=0.0`. It still will not be perfect. That is why week 3 does not
stop at fixing the prompt but builds **code that parses, validates and retries**
after the answer comes back.

**EN — 3.** The department model is small and generally fast, but it slows down
under concurrent load — bear in mind that 30 people are calling it at once
during class.

</details>

---
## 정리 · Summary

- OpenAI 호환 규격 덕분에 **주소·모델·키만 바꾸면** 서버를 교체할 수 있다
- 소형 모델은 **형식 지시를 자주 어긴다** → 3주차의 주제
- 모든 노트북 위쪽의 `PROVIDER` 스위치로 언제든 백업 경로를 쓸 수 있다

> **EN**
> - Thanks to the OpenAI-compatible protocol, **changing the URL, model and key**
>   is enough to swap servers
> - Small models **break format instructions often** → the subject of week 3
> - The `PROVIDER` switch at the top of every notebook gives you the fallback
>   route whenever you need it

**다음 노트북** → [`04_first_mini_agent.ipynb`](https://colab.research.google.com/github/zoomer1975-boop/ai_agent/blob/main/week01/04_first_mini_agent.ipynb)
드디어 '에이전트'를 만듭니다.

> **Next notebook** — at last, we build an *agent*.